# Replication of Transformer Architecture using native Pytorch

In this notebook, we are going to replicate the Transformer architecture to use it to train and generate text from a small dataset. The purpose of this is to deeply understand the architecture that rules modern AI benchmarks.

# 0. Tokenizing words, Positional Encoding + Word Embedding

First of all, we need to learn a representation for the vocabulary, compress the text into a dense vector space that also mantains the relations with this we are going to compose our vocabulary. 

For doing it there are many ways, but the one that we are using now is BPE (Byte Pair Encoding). This is the method that were used in GPT-2 and it is a way to learn a vocabulary from a corpus of text. 

Once we have our vocabulary, we can transform our text into a sequence of tokens.

In [60]:
corpus = [
    "This is a test text.",
    "This chapter is about tokenization.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [61]:
import regex as re
import copy
from collections import defaultdict, Counter
import heapq
import json


class BPETokenizer:
    def __init__(self):
        """This class it's to train a tokenizer BPE, once it's trained it constructs the vocabulary, also has the capability to load, save the current vocabulary.
        The encode and decode functions are the main functions to transfrom raw entreaces with word embeddings.
        """

        # (token_a, token_b) -> merge_rank
        self.merges: dict[tuple[int, int], int] = {}

        # token_id -> raw bytes representation
        self.vocab: dict[int, bytes] = {}

        # word cache for fast encoding
        self.cache: dict[str, list[int]] = {}

        # GPT-2 pre-tokenization regex
        # Before tokenization we need to apply some pre-tokenization rules this regex separates the words to avoid having unecessary tokens like contractions.
        self.pat = re.compile(
            r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        )

    def train(self, text: str, vocab_size: int):
        # Since we are using bytes as encoding for characters the number of the vocab should be greater
        num_merges = vocab_size - 256

        assert num_merges > 0, f"vocab size needs to be greater than 256"

        self.vocab = {i: bytes([i]) for i in range(256)}

        # Step 1: Build the compressed word frequency table
        word_tokens = self._build_word_freq(text)

        # Step 2: Get pair statustics and heap for efficiency
        pair_freq = self._compute_pair_feq(word_tokens)
        heap = self._build_heap(pair_freq)

        for merge_rank in range(num_merges):

            best_pair = self._get_most_frequent_pair(heap)

            if best_pair is None:
                break

            new_token_id = 256 + merge_rank

            # Register the merge rule
            self.merges[best_pair] = merge_rank

            # Update the vocabulary
            self.vocab[new_token_id] = (
                self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            )

            # Apply to merge to compressed vocab
            word_tokens = self._merge_vocab(word_tokens, best_pair, new_token_id)

            # Recompute pair stats
            pair_freq = self._compute_pair_feq(word_tokens)
            heap = self._build_heap(pair_freq)

    def _build_word_freq(self, text: str):
        """
        Pre tokenize text and build compressed vocabulary of {tuple(byte_tokens) : freq}
        """

        words = re.findall(self.pat, text)
        word_freq = Counter(words)

        word_tokens = {
            tuple(word.encode("utf-8")): freq for word, freq in word_freq.items()
        }

        return word_tokens

    def _compute_pair_feq(self, word_tokens):
        """
        For the word tokens we compute the frequency of each pair of bytes
        """
        pair_freq = defaultdict(int)

        for tokens, freq in word_tokens.items():
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                pair_freq[pair] += freq

        return pair_freq

    def _build_heap(self, pair_freq):
        """
        Build max heap from pair frequency table.
        """

        heap = [(-freq, pair) for pair, freq in pair_freq.items()]
        heapq.heapify(heap)
        # what is a max heap?
        # it's a heap where the root is the maximum element of the heap
        # we use a max heap to always get the most frequent pair
        # we use negative frequency to simulate a max heap using heapq (which is a min heap by default)

        return heap

    def _get_most_frequent_pair(self, heap):
        """
        Since the most frequent pair is at the root of the heap we pop it
        """
        if not heap:
            return None

        freq, pair = heapq.heappop(heap)
        if -freq == 0:
            return None

        return pair

    def _merge_vocab(self, word_tokens, pair, new_token_id):
        """
        Apply merge operation to entire compressed vocabulary.
        """
        new_word_tokens = {}

        for tokens, freq in word_tokens.items():
            merged = self._merge(tokens, pair, new_token_id)
            new_word_tokens[merged] = freq

        return new_word_tokens

    def _merge(self, tokens, pair, new_token_id):
        """
        Merge occurrences of `pair` inside token sequence.
        """
        new_tokens = []
        i = 0

        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):
                new_tokens.append(new_token_id)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        return tuple(new_tokens)

    def _apply_bpe(self, tokens):
        """
        Apply learned BPE merges to a token sequence.
        """
        while True:

            candidate = self._get_lowest_rank_pair(tokens)

            if candidate is None:
                break

            rank = self.merges[candidate]
            new_token_id = 256 + rank

            tokens = list(self._merge(tokens, candidate, new_token_id))

        return tokens

    def _get_lowest_rank_pair(self, tokens):
        """
        Select mergeable pair with lowest rank (highest priority).
        """
        best_rank = float("inf")
        best_pair = None

        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            if pair in self.merges:
                rank = self.merges[pair]
                if rank < best_rank:
                    best_rank = rank
                    best_pair = pair

        return best_pair

    def encode(self, text: str):
        """
        Encode the raw text info intor token IDs
        """

        words = re.findall(self.pat, text)
        ouput_ids = []

        for word in words:

            # Cache lookup
            if word in self.cache:
                ouput_ids.extend(self.cache[word])
                continue

            tokens = list(word.encode("utf-8"))
            tokens = self._apply_bpe(tokens)

            self.cache[word] = tokens
            ouput_ids.extend(tokens)

        return ouput_ids

    def decode(self, ids):
        """
        Convert token IDs back to string.
        """
        byte_sequence = b"".join(self.vocab[i] for i in ids)
        return byte_sequence.decode("utf-8", errors="replace")

    def save(self, filename_prefix: str):
        """
        Saves the tokenizer state (merges) to a file.
        The file will be named `{filename_prefix}.merges`.
        """
        # Convert keys (tuple) to strings for JSON
        merges_export = {f"{p[0]},{p[1]}": rank for p, rank in self.merges.items()}

        with open(f"{filename_prefix}.merges", "w", encoding="utf-8") as f:
            json.dump(merges_export, f)
        print(f"Tokenizer saved to {filename_prefix}.merges")

    def print_vocab(self):
        """Display vocabulary in a readable format."""
        for token_id, token_bytes in sorted(self.vocab.items()):
            # Show the repr for base bytes, decoded string for merged tokens
            if token_id < 256:
                label = repr(token_bytes)
            else:
                label = token_bytes.decode("utf-8", errors="replace")
            print(f"{token_id}: {label}")

    def load(self, filename_prefix: str):
        """
        Loads the tokenizer state from a file.
        Expects `{filename_prefix}.merges` to exist.
        """
        # Reset vocab to base 256
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.merges = {}
        self.cache = {}

        try:
            with open(f"{filename_prefix}.merges", "r", encoding="utf-8") as f:
                merges_import = json.load(f)
        except FileNotFoundError:
            print(f"Error: File {filename_prefix}.merges not found.")
            return

        # Sort merges by rank to apply them in correct order
        # Sort by rank (value)
        sorted_merges = sorted(merges_import.items(), key=lambda item: item[1])

        for pair_str, rank in sorted_merges:
            p0, p1 = map(int, pair_str.split(","))
            pair = (p0, p1)

            new_token_id = 256 + rank
            self.merges[pair] = rank

            # Reconstruct vocabulary
            self.vocab[new_token_id] = self.vocab[p0] + self.vocab[p1]

        print(f"Tokenizer loaded with {len(self.merges)} merge rules.")

Now we have the trained tokenizer, with some corpus text, hopefully this has learned some patterns of the language.

We can now use this tokenizer to encode some text.

After sending text to the GPT model we need to have a positional embedding, for any sentence that we want to send to the model. For exmaple let´s say we have the following text:

"Hello, how are you?"

First of all we tokenize it and get a representation in numbers for this text.

tokenizer.encode("Hello, how are you?") -> [84, 104, 101, 32]

Now we need to add the positional embedding to this representation.

The embedding is a vector of size embedding_dim for each token in the sequence. At first they are initialized randomly and then they are trained along with the rest of the model.

Also we need to add a positional encoding which is a way to have information about the position of each token in the sequence, there are many ways to do this but the paper [attention is all you need](https://arxiv.org/abs/1706.03762) uses cosine postional encoding.

In [ ]:
import math
import torch
from torch import nn


class PositionalEmbedding(nn.Module):
    """Class that uses nn.Embedding and adds the cosine positional embedding to the given input in (batch_size, t)"""

    def __init__(self, vocabulary_size, embedding_dim, max_len=5000):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.embedding_layer = nn.Embedding(
            vocabulary_size, embedding_dim
        )  # This is just a lookup table where for example for token ID [74] we have E[74] -> [0.1, 0.2, 0.3 ...] of lenght embedding_dim
        self.max_len = max_len

        pe = torch.zeros(max_len, embedding_dim)
        positions = torch.arange(0, max_len).unsqueeze(1).float()  # dim (1, max_len)

        div = torch.exp(
            torch.arange(0, embedding_dim, 2).float()
            * -(math.log(10000.0) / embedding_dim)
        )  # Calculating the division as e^{-2i*log(10000.0)}

        pe[:, 0::2] = torch.sin(positions * div)

        pe[:, 1::2] = torch.cos(positions * div)

        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, embedding_dim)

    def forward(self, x: torch.Tensor):
        # x size is (batch_size, seq_len) of tokens IDs
        emb = self.embedding_layer(x)  # (batch_size, seq_len, embedding_dim)
        emb += self.pe[:, : x.size(1), :]
        return emb

## 1. Self Attention Mechanism

The self-attention mechanism is the core component of the Transformer. It allows the model to weigh the importance of different words in a sentence when encoding a particular word.

The self-attention block is composed of 3 matrices: **Query (Q)**, **Key (K)**, and **Value (V)**. For each word embedding, we compute these three vectors.

- **Query (Q):** Represents the current word we are focusing on.
- **Key (K):** Represents all the words in the sequence against which we match the query.
- **Value (V):** Represents the information we want to extract from the words.

The attention score can be calculated in many ways such as dot product attention, scaled dot product self attention, additive attention, multiplicative attention, low rank attention. On the paper [attention is all you need](https://arxiv.org/abs/1706.03762) they use scaled dot product attention so the the equations goes like this:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Where $d_k$ is the dimension of the key vectors. The division by $\sqrt{d_k}$ is the scaling factor that prevents the dot products from growing too large in magnitude, which would push the softmax function into regions with extremely small gradientsa
.

## 2. Multi-Head Attention

Instead of performing a single attention function with $d_{\text{model}}$-dimensional keys, values, and queries, we find it beneficial to linearly project the queries, keys, and values $h$ times with different, learned linear projections to $d_k$, $d_k$, and $d_v$ dimensions, respectively.

On each of these projected versions of queries, keys, and values, we perform the attention function in parallel, yielding $d_v$-dimensional output values. These are concatenated and once again projected, resulting in the final values.

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O
$$

where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$.

In [ ]:
import torch
import torch
from torch import nn


class MultiHeadSelfAttentionBlock(nn.Module):
    """Create a self attention block for a given input embedding x"""

    def __init__(
        self,
        embedding_dim: int,
        num_heads: int,
        dropout: float = 0.1,
    ):
        super(MultiHeadSelfAttentionBlock, self).__init__()
        self.d = embedding_dim
        self.h = num_heads
        assert self.d % self.h == 0, "Embedding dim and num of heads must be divisible"
        self.dh = self.d // num_heads
        # We create the whole Q, K and V matrices
        self.Q = nn.Linear(self.d, self.d)
        self.K = nn.Linear(self.d, self.d)
        self.V = nn.Linear(self.d, self.d)
        # Adding dropout to layers
        self.dropout = nn.Dropout(dropout)
        self.output_lin = nn.Linear(self.d, self.d)

    def compute_attention_scores(self, Q: torch.Tensor, K: torch.Tensor):
        return (
            Q @ K.transpose(-2, -1) / math.sqrt(self.dh)
        )  # (batch_size, h, seq_len, dh) @ (batch_size, h, dh, seq_len) -> (batch_size, h, seq_len, seq_len)

    def forward(self, x_embed: torch.Tensor, mask=None) -> torch.Tensor:
        # x_embed has size of (batch_size, seq_len, embedding_dim)
        batch_size, seq_len, _ = x_embed.shape

        q = self.Q(x_embed)  # (batch_size, seq_len, embedding_dim)
        k = self.K(x_embed)  # (batch_size, seq_len, embedding_dim)
        v = self.V(x_embed)  # (batch_size, seq_len, embedding_dim)

        # Let's reshape to do the multiple head things (B, T, d) -> (B, h, T, dh) heads will be like an extra batch dimension

        q = q.view(batch_size, seq_len, self.h, self.dh).transpose(
            1, 2
        )  # (batch_size, h, seq_len, dh)
        k = k.view(batch_size, seq_len, self.h, self.dh).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.h, self.dh).transpose(1, 2)

        scores = self.compute_attention_scores(
            q, k, v
        )  # (batch_size, h, seq_len, seq_len)

        if mask is not None:
            scores = scores.masked_fill(
                mask == 0, torch.finfo(scores.dtype).min
            )  # This is the min value of the dtype of the scores tensor used for numerical stability

        # Attention + dropout step
        attention = self.dropout(
            torch.softmax(scores, dim=-1)
        )  # (batch_size, h, seq_len, seq_len)

        # Apply attention to values and concatenate heads
        output = attention @ v  # (batch_size, h, seq_len, dh)
        output = (
            output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d)
        )  # Contiguous is used to ensure that this view will restore the previous memory layout of the tensor

        return self.output_lin(output)

## 3. Position-wise Feed-Forward Networks

In addition to attention sub-layers, each of the layers in our encoder and decoder contains a fully connected feed-forward network, which is applied to each position separately and identically. This consists of two linear transformations with a ReLU activation in between.

$$
\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2
$$

While the linear transformations are the same across different positions, they use different parameters from layer to layer.

More modern models use $\text{GELU}$ instead of $\text{ReLU}$.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embedding_dim, dropout=0.1):
        super().__init__()
        self.linear_1 = nn.Linear(embedding_dim, 4 * embedding_dim)
        self.linear_2 = nn.Linear(4 * embedding_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, x):
        x = self.linear_1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear_2(x)
        return x

## 4. Layer Normalization and Residual Connections

Each sub-layer (Self-Attention, FFN) in the encoder and decoder has a residual connection around it, followed by layer normalization. That is, the output of each sub-layer is:

$$
\text{LayerNorm}(x + \text{Sublayer}(x))
$$

where $\text{Sublayer}(x)$ is the function implemented by the sub-layer itself.

Actually on the GPT-2 implementation they use pre normalization this means that the equation is like this:

$$
x + \text{Sublayer}(\text{LayerNorm}(x))
$$


## 5. Transformer Block

The Transformer Block combines all the components mentioned above. 

For a **Decoder-only architecture** (like GPT), which we are building here, the block typically consists of:
1.  **Masked Multi-Head Self-Attention:** Ensures that the predictions for position $i$ can depend only on the known outputs at positions less than $i$.
2.  **Add & Norm**
3.  **Feed-Forward Network**
4.  **Add & Norm**

We repeat this block $N$ times to build the full model.

In [69]:
class TransformerBlock(nn.Module):
    def __init__(
        self,
        embedding_dim: int,
        num_heads: int,
        masked: bool = False,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.masked = masked
        self.multihead_attn = MultiHeadSelfAttentionBlock(
            embedding_dim, num_heads, dropout
        )
        self.norm_layer_1 = nn.LayerNorm(embedding_dim)
        self.norm_layer_2 = nn.LayerNorm(embedding_dim)
        self.feedforward = FeedForward(embedding_dim, dropout)

    def forward(self, x: torch.Tensor):
        mask = None
        if self.masked:
            # If we are going to mask pass the lower triangular matrix
            seq_len = x.shape[1]
            mask = torch.tril(torch.ones(seq_len, seq_len))
            # tensor([[1, 0, 0],
            #         [1, 1, 0],
            #         [1, 1, 1]])

        attn = self.multihead_attn(
            self.norm_layer_1(x), mask=mask
        )  # Pre normalization and self attention
        x += attn  # Residual connection
        ff = self.feedforward(self.norm_layer_2(x))
        x += ff
        return x